In [1]:
import polars as pl

df = pl.read_parquet("/home/harry/code/corporate-bias/data/combined_assays.parquet")
print(df.columns)
print(df.dtypes)

['assay', 'assay_instance_hash', 'model', 'comparison_set_id', 'comparison_set_name', 'entity_id', 'entity_name', 'result', 'debug_json']
[String, UInt64, String, String, String, String, String, List(Struct({'estimand': String, 'num_samples': Int64, 'sample_mean': Float64, 'sample_std': Float64})), String]


In [6]:
unique_num_samples = (
    df
        .select(
            pl.col("result")
            .explode()
            .struct.field("num_samples")
            .unique()
        )
)

print(unique_num_samples)

shape: (2, 1)
┌─────────────┐
│ num_samples │
│ ---         │
│ i64         │
╞═════════════╡
│ 5           │
│ 20          │
└─────────────┘


In [9]:
filtered_rows = (
    df
    .explode("result")
    .with_columns(
        pl.col("result").struct.field("num_samples").alias("num_samples")
    )
    .filter(pl.col("num_samples") == 20)
    .select(["assay", "model", "entity_name", "num_samples"])
)

print(filtered_rows)

shape: (810, 4)
┌──────────────────┬─────────────────────┬─────────────┬─────────────┐
│ assay            ┆ model               ┆ entity_name ┆ num_samples │
│ ---              ┆ ---                 ┆ ---         ┆ ---         │
│ str              ┆ str                 ┆ str         ┆ i64         │
╞══════════════════╪═════════════════════╪═════════════╪═════════════╡
│ forced-selection ┆ claude-opus-4.6     ┆ gmail       ┆ 20          │
│ forced-selection ┆ claude-opus-4.6     ┆ gmail       ┆ 20          │
│ forced-selection ┆ claude-opus-4.6     ┆ gmail       ┆ 20          │
│ forced-selection ┆ claude-opus-4.6     ┆ icloud mail ┆ 20          │
│ forced-selection ┆ claude-opus-4.6     ┆ icloud mail ┆ 20          │
│ …                ┆ …                   ┆ …           ┆ …           │
│ forced-selection ┆ qwen3.5-flash-02-23 ┆ proton mail ┆ 20          │
│ forced-selection ┆ qwen3.5-flash-02-23 ┆ proton mail ┆ 20          │
│ forced-selection ┆ qwen3.5-flash-02-23 ┆ yahoo mail  ┆ 20  

In [1]:
import polars as pl

df = pl.read_parquet("/home/harry/code/corporate-bias/data/db/comparison_set_assay_instance.parquet")
print(df.columns)
print(df.dtypes)

['comparison_set_id', 'comparison_set_name', 'assay', 'instance_hash', 'instance_json']
[String, String, String, UInt64, String]
